# 공차 전체 음료 칼로리 분석

- **수집 대상:** https://www.gong-cha.co.kr/brand/menu/product?category=001001
- **robots.txt 확인:** `/brand/menu/` 경로는 수집 허용 (`disallow: /thdadmin/, /upload/, /html2/, /html/`)
- **목적:** 공차 음료 중 칼로리가 가장 높은/낮은 음료 Top 5 찾기 (**L 사이즈 아이스(Cold) 기준**)

### 사이트 구조 (직접 확인한 내용)
- 음료 탭(New 시즌 메뉴, 밀크티, 커피 …)은 클릭하면 주소의 `category` 번호만 바뀌는 **정적 페이지**예요.
  → Selenium 없이 `requests` + `BeautifulSoup`으로 충분해요.
- 상세 페이지 영양정보 표의 열 순서: `구분(Cold/Hot) | 사이즈(L/J) | 컵 용량(ml) | 열량(kcal) | 당류 | 단백질 | 포화지방 | 나트륨 | 카페인`

**셀을 위에서부터 순서대로 실행하세요** (`Shift + Enter`).

## 0. 라이브러리 설치 (처음 한 번만)

In [ ]:
%pip install requests beautifulsoup4 pandas

## 1. 라이브러리 불러오기 + 설정

In [ ]:
import re
import time

import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE = "https://www.gong-cha.co.kr"
LIST_URL = BASE + "/brand/menu/product?category={code}"
HEADERS = {"User-Agent": "Mozilla/5.0"}

# 음료 탭 이름과 category 번호 (사이트의 탭 HTML에서 확인한 값)
# 먼저 나온 탭 이름이 그 음료의 카테고리가 되도록 진짜 종류를 앞에 뒀어요.
TABS = {
    "001006": "밀크티",
    "001010": "스무디",
    "001003": "오리지널 티",
    "001015": "프룻티&모어",
    "001011": "커피",
    "001017": "요거티",
    "001001": "New 시즌 메뉴",
    "001002": "베스트셀러",
}

## 2. 함수: 탭(카테고리) 페이지에서 메뉴 이름과 상세 링크 모으기

In [ ]:
def get_menus_in_tab(code):
    res = requests.get(LIST_URL.format(code=code), headers=HEADERS, timeout=10)
    res.raise_for_status()
    soup = BeautifulSoup(res.text, "html.parser")

    items = []
    # 메뉴 하나 = <a href="/brand/menu/product_detail?category=...&no=..."> 안의 <p>메뉴이름</p>
    for a in soup.select('a[href*="product_detail"]'):
        name = " ".join(a.get_text(" ", strip=True).split())
        url = BASE + a["href"]               # 링크 주소를 그대로 사용 (category 번호 포함)
        if name:
            items.append({"name": name, "url": url})
    return items

## 3. 함수: 상세 페이지 영양정보 표에서 L 사이즈 아이스(Cold) 값 가져오기

In [ ]:
def get_nutrition(url):
    res = requests.get(url, headers=HEADERS, timeout=10)
    res.raise_for_status()
    soup = BeautifulSoup(res.text, "html.parser")

    rows = []
    temp = None                                  # Cold / Hot
    for tr in soup.select("table tbody tr"):
        cells = [td.get_text(strip=True) for td in tr.find_all("td")]
        if len(cells) < 8:
            continue
        # 표에서 'Cold'/'Hot' 칸은 여러 줄을 합쳐 놓아서(rowspan) 첫 줄에만 있어요.
        # 그래서 칸이 9개면 첫 칸이 온도, 8개면 위 줄의 온도를 그대로 써요.
        if len(cells) == 9:
            temp = cells[0]
        # 뒤에서부터 세면 칸 위치가 항상 같아요:
        # [-8]사이즈 [-7]용량 [-6]열량 [-5]당류 [-4]단백질 [-3]포화지방 [-2]나트륨 [-1]카페인
        rows.append({
            "온도": temp,
            "사이즈": cells[-8],
            "용량(ml)": float(cells[-7]),
            "칼로리(kcal)": float(cells[-6]),
            "당류(g)": float(cells[-5]),
            "카페인(mg)": float(cells[-1]),
        })

    # L 사이즈 아이스(Cold) 줄 찾기 → 없으면 None
    for r in rows:
        if r["온도"] == "Cold" and r["사이즈"] == "L":
            return r
    return None

## 4. 1단계 실행: 모든 탭에서 음료 모으기 (중복 제거)

In [ ]:
menus = {}                                   # {메뉴이름: 정보}  → 이름이 같으면 한 번만
for code, tab in TABS.items():
    items = get_menus_in_tab(code)
    new = 0
    for it in items:
        if it["name"] not in menus:
            menus[it["name"]] = {"name": it["name"], "category": tab, "url": it["url"]}
            new += 1
    print(f"[{tab}] {len(items)}개 (새로 추가 {new}개)")
    time.sleep(1)                            # 서버 부담 줄이기

menus = list(menus.values())
print(f"\n중복 빼고 음료 {len(menus)}개")

## 5. 2단계 실행: 각 음료의 영양정보 모으기
음료 하나당 1초씩 쉬어서 2분 정도 걸려요.

In [ ]:
for i, m in enumerate(menus, 1):
    try:
        info = get_nutrition(m["url"])
    except Exception as e:
        print("  에러:", m["name"], e)
        info = None
    if info:
        m.update(info)
        print(f"[{i}/{len(menus)}] {m['name']}: {info['칼로리(kcal)']:.0f}kcal")
    else:
        print(f"[{i}/{len(menus)}] {m['name']}: L 사이즈 아이스 정보 없음 (제외)")
    time.sleep(1)

## 6. 데이터 확인 + CSV 저장

In [ ]:
df = pd.DataFrame(menus)
df.to_csv("gongcha_all_drinks.csv", index=False, encoding="utf-8-sig")  # 엑셀에서 한글 안 깨지게
print("CSV 저장 완료: gongcha_all_drinks.csv")
df

## 7. 분석: 칼로리 높은/낮은 음료 Top 5 (L 사이즈 아이스 기준)

In [ ]:
data = df.dropna(subset=["칼로리(kcal)"])   # L 사이즈 아이스 정보가 없는 음료는 제외
top5 = data.sort_values("칼로리(kcal)", ascending=False, kind="stable").head(5)
low5 = data.sort_values("칼로리(kcal)", ascending=True, kind="stable").head(5)

print("[칼로리 높은 음료 Top 5] (L 사이즈 아이스 기준)")
for rank, (name, kcal) in enumerate(zip(top5["name"], top5["칼로리(kcal)"]), 1):
    print(f"{rank}등. {name} {kcal:.0f}kcal")

print("\n[칼로리 낮은 음료 Top 5] (L 사이즈 아이스 기준)")
for rank, (name, kcal) in enumerate(zip(low5["name"], low5["칼로리(kcal)"]), 1):
    print(f"{rank}등. {name} {kcal:.0f}kcal")

print(f"\n분석한 음료 {len(data)}개, 평균 칼로리 {data['칼로리(kcal)'].mean():.0f}kcal")